In [ ]:
from ase import io
from ase.io import Trajectory
#from pfcc_extras.visualize.view import view_ngl
import numpy as np
import os

# Helper to count bound O atoms
def count_bound_o(atoms, cutoff=3.5):
    syms = atoms.get_chemical_symbols()
    pos  = atoms.get_positions()
    o_idx     = [i for i,s in enumerate(syms) if s=='O']
    metal_idx = [i for i,s in enumerate(syms) if s in ('Zr','V')]
    if not o_idx or not metal_idx: return 0
    return sum(1 for oi in o_idx
               if min(np.linalg.norm(pos[oi]-pos[mj]) for mj in metal_idx) < cutoff)

print('Imports done')

## Parameters — edit these

In [ ]:
# Set the directory for the system you want to view
directory = 'C://Users/danie/Downloads/jovyan(1)/jovyan/Matlantis/VN_111/873K'

# Which rounds to show as end-of-round snapshots
# Use None to show all available rounds
show_rounds = None  # e.g. [0, 4, 8, 15] for specific rounds, or None for all

# Which round trajectory to play as a movie (set to None to skip)
movie_round = 0  # e.g. 0 for first round, 15 for last round

## View end-of-round snapshots

Shows the surface structure after each round of O addition — equivalent to Ji's Figures 2-10.

In [ ]:
# Load all end-of-round structures
snapshots = []
labels = []
i = 0
while True:
    path = os.path.join(directory, f'{i:0>2d}_MD.traj')
    if not os.path.exists(path):
        break
    if show_rounds is None or i in show_rounds:
        atoms = io.read(path, index=-1)
        snapshots.append(atoms)
        bound = count_bound_o(atoms)
        syms = atoms.get_chemical_symbols()
        n_o = syms.count('O')
        labels.append(f'Round {i}: {n_o} O total, {bound} bound')
    i += 1

print(f'Loaded {len(snapshots)} snapshots')
for l in labels:
    print(f'  {l}')

# View as slideshow — use arrow keys to step through rounds
view_ngl(snapshots)

## View full concatenated trajectory

Combines all round trajectories into one continuous movie showing the entire oxidation process.

In [ ]:
# Concatenate all MD trajectory files into one
all_frames = []
round_boundaries = []  # frame indices where each round starts
i = 0
while True:
    path = os.path.join(directory, f'{i:0>2d}_MD-trajectory.traj')
    if not os.path.exists(path):
        break
    try:
        frames = io.read(path, index=':')
        round_boundaries.append(len(all_frames))
        all_frames.extend(frames)
        print(f'Round {i:02d}: {len(frames)} frames')
    except Exception as e:
        print(f'Round {i:02d}: error - {e}')
    i += 1

print(f'\nTotal frames: {len(all_frames)}')
print(f'Round boundaries at frames: {round_boundaries}')

# Play as movie
view_ngl(all_frames)

## View single round as movie

Play the MD trajectory for one specific round.

In [ ]:
if movie_round is not None:
    path = os.path.join(directory, f'{movie_round:0>2d}_MD-trajectory.traj')
    if os.path.exists(path):
        frames = io.read(path, index=':')
        print(f'Round {movie_round}: {len(frames)} frames')
        
        # Print bound O at start and end
        print(f'Start: {frames[0].get_chemical_formula()}, bound O: {count_bound_o(frames[0])}')
        print(f'End:   {frames[-1].get_chemical_formula()}, bound O: {count_bound_o(frames[-1])}')
        
        view_ngl(frames, representations=['ball+stick'])
    else:
        print(f'File not found: {path}')
else:
    print('movie_round is None — skipping single round movie')

## View final relaxed structure

The fully relaxed saturated surface — equivalent to Ji's final structure in each figure.

In [ ]:
final_path = os.path.join(directory, 'final_relaxed.traj')
if os.path.exists(final_path):
    atoms = io.read(final_path, index=-1)
    syms = atoms.get_chemical_symbols()
    bound = count_bound_o(atoms)
    print(f'Final relaxed: {atoms.get_chemical_formula()}')
    print(f'Bound O: {bound}')
    print(f'N lost: {36 - syms.count("N") if "Zr" in syms else 64 - syms.count("N")}')
    view_ngl([atoms], representations=['ball+stick'])
else:
    print(f'No final_relaxed.traj found in {directory}')
    print('Run the relaxation cell in Etch-run_Ji-protocol.ipynb first')

## Bound O progression plot

Plots bound O count vs round, comparable to Ji's oxygen coverage data.

## Multi-seed oxidation plot

Shows all seed runs overlaid with mean ± std band.
Set `base_directory` = parent folder containing `seed_1`, `seed_2` etc.
Automatically detects system and temperature from the path.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
from ase import io

# ── Parameters ───────────────────────────────────────────────────────────────
# Base directory for the system (parent of seed_1, seed_2 ... folders)
base_directory = directory  # uses whatever directory is set in cell 2
n_seeds        = 5          # number of seed runs to look for

JI_REFERENCE = {
    'ZrN_100': {'295': 8,  '363': 12, '1023': 28},
    'ZrN_110': {'295': 18, '363': 20, '1023': 32},
    'VN_111':  {'295': 11, '363': 18, '873':  12},
}

def count_bound_o(atoms, cutoff=3.5):
    syms = atoms.get_chemical_symbols()
    pos  = atoms.get_positions()
    o_idx     = [i for i,s in enumerate(syms) if s=='O']
    metal_idx = [i for i,s in enumerate(syms) if s in ('Zr','V')]
    if not o_idx or not metal_idx: return 0
    return sum(1 for oi in o_idx
               if min(np.linalg.norm(pos[oi]-pos[mj]) for mj in metal_idx) < cutoff)

def load_seed_data(seed_dir):
    """Load bound O progression from one seed directory."""
    bound, total_o = [], []
    i = 0
    while True:
        path = os.path.join(seed_dir, f'{i:0>2d}_MD.traj')
        if not os.path.exists(path): break
        try:
            atoms = io.read(path, index=-1)
            syms  = atoms.get_chemical_symbols()
            # Only include round if slab is still healthy
            metal_idx = [j for j,s in enumerate(syms) if s in ('Zr','V')]
            pos = atoms.get_positions()
            metal_z = pos[metal_idx, 2]
            if (metal_z.max() - metal_z.min() < 30 and
                    pos[:,2].max() < 500 and len(atoms) > 50):
                bound.append(count_bound_o(atoms))
                total_o.append(syms.count('O'))
            else:
                print(f'  Seed {seed_dir}: slab crashed at round {i} — stopping')
                break
        except Exception as e:
            print(f'  Round {i} error: {e}')
            break
        i += 1
    return bound, total_o

# ── Load all seeds ────────────────────────────────────────────────────────────
all_bound = {}  # seed -> list of bound O per round
for s in range(1, n_seeds + 1):
    seed_dir = os.path.join(base_directory, f'seed_{s}')
    if os.path.exists(seed_dir):
        b, _ = load_seed_data(seed_dir)
        if b:
            all_bound[s] = b
            print(f'Seed {s}: {len(b)} rounds, final bound O = {b[-1]}')
        else:
            print(f'Seed {s}: no data found')
    else:
        print(f'Seed {s}: directory not found ({seed_dir})')

if not all_bound:
    print('No seed data found. Make sure seed_1..seed_N subdirectories exist.')
else:
    # ── Pad to common length for averaging ───────────────────────────────────
    max_len = max(len(v) for v in all_bound.values())
    padded  = np.full((len(all_bound), max_len), np.nan)
    for i, (s, b) in enumerate(all_bound.items()):
        padded[i, :len(b)] = b

    mean_bound = np.nanmean(padded, axis=0)
    std_bound  = np.nanstd(padded, axis=0)
    rounds     = np.arange(1, max_len + 1)

    # ── Guess system and temperature from directory path ──────────────────────
    parts  = base_directory.replace('\\', '/').split('/')
    system = next((p for p in parts if any(x in p for x in
                   ('ZrN_100','ZrN_110','VN_111'))), 'unknown')
    temp   = next((p[:-1] for p in parts if p.endswith('K')
                   and p[:-1].isdigit()), 'unknown')
    ji_val = JI_REFERENCE.get(system, {}).get(temp)

    # ── Plot ──────────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 5))

    # Individual seed traces (light)
    colors = plt.cm.Blues(np.linspace(0.35, 0.75, len(all_bound)))
    for (s, b), col in zip(all_bound.items(), colors):
        ax.plot(range(1, len(b)+1), b, color=col, alpha=0.4,
                linewidth=1, label=f'Seed {s}')

    # Mean line and shaded ±1 std band
    ax.plot(rounds, mean_bound, 'o-', color='steelblue',
            linewidth=2.5, markersize=5, label='Mean', zorder=5)
    ax.fill_between(rounds,
                    mean_bound - std_bound,
                    mean_bound + std_bound,
                    alpha=0.25, color='steelblue', label='±1 std')

    # Ji reference line
    if ji_val:
        ax.axhline(ji_val, color='red', linestyle='--', linewidth=1.5,
                   label=f'Ji DFT ({ji_val} bound O)')

    ax.set_xlabel('Round', fontsize=12)
    ax.set_ylabel('Bound O atoms', fontsize=12)
    ax.set_title(f'{system} {temp}K — Oxidation progression ({len(all_bound)} seeds)',
                 fontsize=13)
    ax.legend(fontsize=9, ncol=2)
    ax.grid(True, alpha=0.3)

    # Stats box
    final_vals = [b[-1] for b in all_bound.values()]
    stats_txt = (f'Final bound O:\n'
                 f'  Mean: {np.mean(final_vals):.1f}\n'
                 f'  Std:  {np.std(final_vals):.1f}\n'
                 f'  Range: {min(final_vals)}-{max(final_vals)}')
    if ji_val:
        stats_txt += f'\n  Ji DFT: {ji_val}'
    ax.text(0.02, 0.97, stats_txt, transform=ax.transAxes,
            verticalalignment='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    plt.tight_layout()
    out = os.path.join(base_directory, f'{system}_{temp}K_seeds.png')
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved to {out}')


In [ ]:
import matplotlib.pyplot as plt

# Ji reference values
JI_REFERENCE = {
    'ZrN_100': {'295': 8,  '363': 12, '1023': 28},
    'ZrN_110': {'295': 18, '363': 20, '1023': 32},
    'VN_111':  {'295': 11, '363': 18, '873':  12},
}

# Load bound O progression from saved round structures
rounds_list = []
bound_o_list = []
n_o_list = []
i = 0
while True:
    path = os.path.join(directory, f'{i:0>2d}_MD.traj')
    if not os.path.exists(path):
        break
    atoms = io.read(path, index=-1)
    syms = atoms.get_chemical_symbols()
    rounds_list.append(i + 1)
    bound_o_list.append(count_bound_o(atoms))
    n_o_list.append(syms.count('O'))
    i += 1

# Guess system and temperature from directory
parts = directory.replace('\\', '/').split('/')
system = next((p for p in parts if any(x in p for x in ('ZrN_100','ZrN_110','VN_111'))), 'unknown')
temp   = next((p[:-1] for p in parts if p.endswith('K') and p[:-1].isdigit()), 'unknown')
ji_val = JI_REFERENCE.get(system, {}).get(temp)

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(rounds_list, bound_o_list, 'o-', color='steelblue', label='PFP bound O')
ax.plot(rounds_list, n_o_list, 's--', color='lightblue', alpha=0.5, label='Total O in slab')
if ji_val:
    ax.axhline(ji_val, color='red', linestyle='--', label=f'Ji DFT ({ji_val} bound O)')
ax.set_xlabel('Round')
ax.set_ylabel('O atoms')
ax.set_title(f'{system} {temp}K — Oxidation progression')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(directory, f'{system}_{temp}K_oxidation.png'), dpi=150)
plt.show()
print(f'Saved to {system}_{temp}K_oxidation.png')